In [1]:
# =============================================================================
# 02_Gold_Analytics.py — SA Retail Analytics
# Gold Layer: Business Analytics & Aggregations
#
# Purpose: Read Silver Delta tables → aggregate → write Gold Delta tables
# Attach Lakehouse: SalesStorage
# =============================================================================

StatementMeta(, 9f72e0bf-33a1-45be-8c76-53e8ee2d58ac, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries loaded!")

StatementMeta(, 9f72e0bf-33a1-45be-8c76-53e8ee2d58ac, 4, Finished, Available, Finished, False)

Libraries loaded!


In [3]:
# Read Silver Delta Tables
df_customers = spark.read.format("delta").table("silver_customer")
df_products = spark.read.format("delta").table("silver_product")
df_orders = spark.read.format("delta").table("silver_orders")

print(f"silver_customer: {df_customers.count()} rows")
print(f"silver_product: {df_products.count()} rows")
print(f"silver_orders: {df_orders.count()} rows")

StatementMeta(, 9f72e0bf-33a1-45be-8c76-53e8ee2d58ac, 5, Finished, Available, Finished, False)

silver_customer: 20 rows
silver_product: 20 rows
silver_orders: 352 rows


In [4]:
# gold_customer
# Total orders, revenue, items, and average order value per customer
# Joined with customer demographics

order_summary = (
    df_orders
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("line_total"), 2).alias("total_revenue"),
        F.sum("quantity").alias("total_items"),
        F.round(F.avg("order_total"), 2).alias("avg_order_value"),
        F.min("order_date").alias("first_order_date"),
        F.max("order_date").alias("last_order_date"),
    )
)
 
gold_customer = (
    df_customers
    .join(order_summary, on="customer_id", how="left")
    .withColumn("revenue_rank", F.dense_rank().over(
        Window.orderBy(F.col("total_revenue").desc())
    ))
    .fillna(0, subset=["total_orders", "total_revenue", "total_items", "avg_order_value"])
    .select(
        "customer_id", "name", "email", "city", "province", "segment",
        "total_orders", "total_revenue", "total_items", "avg_order_value",
        "first_order_date", "last_order_date", "revenue_rank"
    )
    .orderBy("revenue_rank")
)
 
print(f"\n gold_customer: {gold_customer.count()} rows")
gold_customer.show(5, truncate=False)

StatementMeta(, 9f72e0bf-33a1-45be-8c76-53e8ee2d58ac, 6, Finished, Available, Finished, False)


 gold_customer: 20 rows
+-----------+-------------+-----------------------+------------+----------+-------+------------+-------------+-----------+---------------+----------------+---------------+------------+
|customer_id|name         |email                  |city        |province  |segment|total_orders|total_revenue|total_items|avg_order_value|first_order_date|last_order_date|revenue_rank|
+-----------+-------------+-----------------------+------------+----------+-------+------------+-------------+-----------+---------------+----------------+---------------+------------+
|C008       |Kagiso Molefe|kagiso.molefe@gmail.com|Bloemfontein|Free State|BUDGET |6           |15140.06     |94         |3049.69        |2025-09-26      |2026-04-03     |1           |
|C014       |Precious Moyo|precious.moyo@gmail.com|Roodepoort  |Gauteng   |BUDGET |8           |10069.08     |92         |1603.73        |2025-07-11      |2026-01-21     |2           |
|C010       |Mpho Tau     |mpho.tau@hotmail.com   

In [5]:
# gold_product
# Sales performance per product

gold_product = (
    df_orders
    .groupBy("product_id")
    .agg(
        F.sum("quantity").alias("total_units_sold"),
        F.round(F.sum("line_total"), 2).alias("total_revenue"),
        F.countDistinct("order_id").alias("order_count"),
        F.round(F.avg("unit_price"), 2).alias("avg_selling_price"),
    )
    .join(df_products.select("product_id", "product_name", "category", "price", "stock"),
          on="product_id", how="left")
    .withColumn("revenue_per_unit", F.round(F.col("total_revenue") / F.col("total_units_sold"), 2))
    .withColumn("product_rank", F.dense_rank().over(
        Window.orderBy(F.col("total_revenue").desc())
    ))
    .select(
        "product_id", "product_name", "category", "price", "stock",
        "total_units_sold", "total_revenue", "order_count",
        "avg_selling_price", "revenue_per_unit", "product_rank"
    )
    .orderBy("product_rank")
)
 
print(f"\n gold_product: {gold_product.count()} rows")
gold_product.show(5, truncate=False)

StatementMeta(, 9f72e0bf-33a1-45be-8c76-53e8ee2d58ac, 7, Finished, Available, Finished, False)


 gold_product: 20 rows
+----------+------------------------+-------------+------+-----+----------------+-------------+-----------+-----------------+----------------+------------+
|product_id|product_name            |category     |price |stock|total_units_sold|total_revenue|order_count|avg_selling_price|revenue_per_unit|product_rank|
+----------+------------------------+-------------+------+-----+----------------+-------------+-----------+-----------------+----------------+------------+
|P013      |Truworths Denim Jeans   |Clothing     |499.99|80   |94              |46999.06     |24         |499.99           |499.99          |1           |
|P014      |Volkano USB Charging Hub|Electronics  |299.99|90   |79              |23699.21     |20         |299.99           |299.99          |2           |
|P007      |Ariel Washing Powder 2kg|Household    |79.99 |220  |71              |5679.29      |21         |79.99            |79.99           |3           |
|P005      |Nescafé Classic 200g    |Gro

In [6]:
# gold_segment
# Aggregation by customer segment (Premium, Regular, Budget)

gold_segment = (
    df_orders
    .join(df_customers.select("customer_id", "segment"), on="customer_id", how="left")
    .groupBy("segment")
    .agg(
        F.countDistinct("customer_id").alias("total_customers"),
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("line_total"), 2).alias("total_revenue"),
        F.round(F.avg("order_total"), 2).alias("avg_order_value"),
        F.sum("quantity").alias("total_items"),
    )
    .withColumn("revenue_pct", F.round(
        F.col("total_revenue") / F.sum("total_revenue").over(Window.orderBy(F.lit(1)).rowsBetween(-999, 999)) * 100,
        1
    ))
    .orderBy(F.col("total_revenue").desc())
)
 
print(f"\n gold_segment: {gold_segment.count()} rows")
gold_segment.show(truncate=False)

StatementMeta(, 9f72e0bf-33a1-45be-8c76-53e8ee2d58ac, 8, Finished, Available, Finished, False)


 gold_segment: 3 rows
+-------+---------------+------------+-------------+---------------+-----------+-----------+
|segment|total_customers|total_orders|total_revenue|avg_order_value|total_items|revenue_pct|
+-------+---------------+------------+-------------+---------------+-----------+-----------+
|BUDGET |5              |35          |41262.34     |1615.35        |366        |35.1       |
|REGULAR|8              |44          |39331.12     |1001.46        |388        |33.5       |
|PREMIUM|7              |41          |36920.4      |1071.36        |460        |31.4       |
+-------+---------------+------------+-------------+---------------+-----------+-----------+



In [7]:
# gold_category
# Performance by product category

gold_category = (
    df_orders
    .join(df_products.select("product_id", "category"), on="product_id", how="left")
    .groupBy("category")
    .agg(
        F.countDistinct("product_id").alias("total_products"),
        F.sum("quantity").alias("total_units_sold"),
        F.round(F.sum("line_total"), 2).alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.avg("line_total"), 2).alias("avg_line_value"),
    )
    .withColumn("category_rank", F.dense_rank().over(
        Window.orderBy(F.col("total_revenue").desc())
    ))
    .orderBy("category_rank")
)
 
print(f"\n gold_category: {gold_category.count()} rows")
gold_category.show(truncate=False)

StatementMeta(, 9f72e0bf-33a1-45be-8c76-53e8ee2d58ac, 9, Finished, Available, Finished, False)


 gold_category: 7 rows
+-------------+--------------+----------------+-------------+------------+--------------+-------------+
|category     |total_products|total_units_sold|total_revenue|total_orders|avg_line_value|category_rank|
+-------------+--------------+----------------+-------------+------------+--------------+-------------+
|Clothing     |2             |117             |50448.83     |31          |1483.79       |1            |
|Electronics  |2             |129             |28198.71     |31          |783.3         |2            |
|Personal Care|3             |194             |11758.06     |44          |226.12        |3            |
|Groceries    |5             |271             |10775.29     |65          |138.14        |4            |
|Household    |3             |185             |8696.15      |50          |152.56        |5            |
|Beverages    |3             |177             |5949.23      |45          |116.65        |6            |
|Snacks       |2             |141       

In [8]:
# gold_monthly_trend
# Monthly revenue trend (for time-series charts in Power Bi)

gold_monthly_trend = (
    df_orders
    .groupBy("order_year", "order_month", "order_quarter")
    .agg(
        F.round(F.sum("line_total"), 2).alias("monthly_revenue"),
        F.countDistinct("order_id").alias("monthly_orders"),
        F.sum("quantity").alias("monthly_units"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.round(F.avg("order_total"), 2).alias("avg_basket_size"),
    )
    .withColumn("month_label",
        F.concat(F.col("order_year").cast("string"), F.lit("-"),
                 F.lpad(F.col("order_month").cast("string"), 2, "0"))
    )
    .orderBy("order_year", "order_month")
)
 
print(f"\n gold_monthly_trend: {gold_monthly_trend.count()} rows")
gold_monthly_trend.show(truncate=False)

StatementMeta(, 9f72e0bf-33a1-45be-8c76-53e8ee2d58ac, 10, Finished, Available, Finished, False)


 gold_monthly_trend: 13 rows
+----------+-----------+-------------+---------------+--------------+-------------+----------------+---------------+-----------+
|order_year|order_month|order_quarter|monthly_revenue|monthly_orders|monthly_units|unique_customers|avg_basket_size|month_label|
+----------+-----------+-------------+---------------+--------------+-------------+----------------+---------------+-----------+
|2025      |4          |2            |314.96         |1             |4            |1               |314.96         |2025-04    |
|2025      |5          |2            |10545.17       |9             |83           |7               |1104.34        |2025-05    |
|2025      |6          |2            |13819.12       |11            |138          |9               |1318.2         |2025-06    |
|2025      |7          |3            |5633.37        |7             |63           |7               |977.78         |2025-07    |
|2025      |8          |3            |5579.1         |7            

In [9]:
# gold_store_performance
# Revenue and orders per retail store

gold_store_performance = (
    df_orders
    .groupBy("store")
    .agg(
        F.round(F.sum("line_total"), 2).alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.round(F.avg("order_total"), 2).alias("avg_basket_size"),
    )
    .withColumn("store_rank", F.dense_rank().over(
        Window.orderBy(F.col("total_revenue").desc())
    ))
    .orderBy("store_rank")
)
 
print(f"\n gold_store_performance: {gold_store_performance.count()} rows")
gold_store_performance.show(truncate=False)

StatementMeta(, 9f72e0bf-33a1-45be-8c76-53e8ee2d58ac, 11, Finished, Available, Finished, False)


 gold_store_performance: 8 rows
+-----------------------+-------------+------------+----------------+---------------+----------+
|store                  |total_revenue|total_orders|unique_customers|avg_basket_size|store_rank|
+-----------------------+-------------+------------+----------------+---------------+----------+
|Shoprite Durban        |23898.98     |18          |12              |1597.43        |1         |
|Checkers Sea Point     |17919.52     |16          |10              |1307.22        |2         |
|Pick n Pay Sandton     |17109.9      |15          |10              |1430.29        |3         |
|Clicks Rosebank        |14120.96     |14          |12              |1025.3         |4         |
|Checkers Menlyn        |13934.51     |17          |12              |1113.09        |5         |
|Woolworths V&A         |11707.87     |14          |8               |1303.81        |6         |
|Pick n Pay Bloemfontein|11574.07     |15          |11              |805.38         |7        

In [10]:
# gold_payment_methods
# Payment method preference

gold_payment_methods = (
    df_orders
    .groupBy("payment_method")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("line_total"), 2).alias("total_revenue"),
        F.countDistinct("customer_id").alias("unique_customers"),
    )
    .withColumn("order_pct", F.round(
        F.col("total_orders") / F.sum("total_orders").over(Window.orderBy(F.lit(1)).rowsBetween(-999, 999)) * 100,
        1
    ))
    .orderBy(F.col("total_orders").desc())
)
 
print(f"\n gold_payment_methods: {gold_payment_methods.count()} rows")
gold_payment_methods.show(truncate=False)

StatementMeta(, 9f72e0bf-33a1-45be-8c76-53e8ee2d58ac, 12, Finished, Available, Finished, False)


 gold_payment_methods: 5 rows
+--------------+------------+-------------+----------------+---------+
|payment_method|total_orders|total_revenue|unique_customers|order_pct|
+--------------+------------+-------------+----------------+---------+
|Cash          |28          |21062.73     |17              |23.3     |
|SnapScan      |26          |45428.79     |14              |21.7     |
|Card          |23          |15480.58     |14              |19.2     |
|EFT           |23          |19177.46     |12              |19.2     |
|Zapper        |20          |16364.3      |12              |16.7     |
+--------------+------------+-------------+----------------+---------+



In [15]:
# Gold Data Tables and export to csv for Power Bi, since i am on a free plan
# I cannot connect the lakehouse directly to Power BI

print("\n Exporting Gold tables to CSV for Power BI...")

export_base_path = "Files/gold"  

tables_export = {
    "gold_customer":          gold_customer,
    "gold_product":           gold_product,
    "gold_segment":           gold_segment,
    "gold_category":          gold_category,
    "gold_monthly_trend":     gold_monthly_trend,
    "gold_store_performance": gold_store_performance,
    "gold_payment_methods":   gold_payment_methods,
}

for name, df in tables_export.items():
    path = f"{export_base_path}/{name}"

    (
        df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .format("csv")  
        .save(path)
    )

    print(f"  {name} exported to {path}")

print("\n CSV export complete — ready for Power BI")

StatementMeta(, 9f72e0bf-33a1-45be-8c76-53e8ee2d58ac, 17, Finished, Available, Finished, False)


 Exporting Gold tables to CSV for Power BI...
  gold_customer exported to Files/gold/gold_customer
  gold_product exported to Files/gold/gold_product
  gold_segment exported to Files/gold/gold_segment
  gold_category exported to Files/gold/gold_category
  gold_monthly_trend exported to Files/gold/gold_monthly_trend
  gold_store_performance exported to Files/gold/gold_store_performance
  gold_payment_methods exported to Files/gold/gold_payment_methods

 CSV export complete — ready for Power BI
